In [1]:
# used the instructions of https://www.sparkcodehub.com/pyspark/use-cases/recommendation-systems

%run ./sparsed_data.ipynb
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.ml.evaluation import RegressionEvaluator



# SparkSession (like the wpo)
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("AnimeRecommender") \
    .config("spark.ui.port", "4040") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .config("spark.driver.extraJavaOptions", "-Duser.name=admin") \
    .config("spark.driver.extraJavaOptions", "-Dsun.security.auth.login.config=C:/dev/null") \
    .getOrCreate()

+-------+--------+-----+
|user_id|anime_id|score|
+-------+--------+-----+
|     32|   34096|  8.0|
|   1104|   34599| 10.0|
|   1825|   28891|  7.0|
|   3796|    2904|  9.0|
|   9589|    4181| 10.0|
|   9872|    2904| 10.0|
|    554|   16664|  6.0|
+-------+--------+-----+
only showing top 7 rows

hello


In [2]:
#pre processed data

reviews_data = spark.read.csv("../data/preprocessed/reviews.csv", header=True, inferSchema=True)
anime_data = spark.read.csv("../data/preprocessed/animes.csv", header=True, inferSchema=True)

# The profiles need to be represented in numbers.
# Tried using cast like the intstructions but didn't work due to 
review_indexed = StringIndexer(inputCol="profile", outputCol="user_id")
reviews_data_indexed = review_indexed.fit(reviews_data).transform(reviews_data)

start_data = reviews_data_indexed.select(
    col("user_id").cast("integer"),
    col("anime_id").cast("integer"),
    col("score").cast("float")
)

# final_data = clean_sparse_data(start_data, min_user_reviews=5, min_anime_reviews=10)



start_data.show(7)

+-------+--------+-----+
|user_id|anime_id|score|
+-------+--------+-----+
|     32|   34096|  8.0|
|   1104|   34599| 10.0|
|   1825|   28891|  7.0|
|   3796|    2904|  9.0|
|   9589|    4181| 10.0|
|   9872|    2904| 10.0|
|    554|   16664|  6.0|
+-------+--------+-----+
only showing top 7 rows



In [9]:
####### Bias #######

def get_global_mean(train_data):
    return train_data.agg(F.avg("score")).collect()[0][0]

def compute_user_biases(train_data, global_mean, lambda_user=5.0):
    return train_data.groupBy("user_id") \
            .agg(
                F.count("score").alias("num_user_reviews"),
                F.sum(F.col("score") - global_mean).alias("sum_user_diff")
            ) \
            .withColumn("user_bias", F.col("sum_user_diff") / (F.col("num_user_reviews") + lambda_user)) \
            .select("user_id", "user_bias")

def compute_item_biases(train_data, global_mean, lambda_item=5.0):
    return train_data.groupBy("anime_id") \
            .agg(
                F.count("score").alias("num_item_reviews"),
                F.sum(F.col("score") - global_mean).alias("sum_item_diff")
            ) \
            .withColumn("item_bias", F.col("sum_item_diff") / (F.col("num_item_reviews") + lambda_item)) \
            .select("anime_id", "item_bias")

def compute_all_biases(train_data, lambda_user=10.0, lambda_item=10.0):
    global_mean = get_global_mean(train_data)
    user_biases = compute_user_biases(train_data, global_mean, lambda_user)
    item_biases = compute_item_biases(train_data, global_mean, lambda_item)
    return global_mean, user_biases, item_biases

In [10]:
def train_and_evaluate_als(train_data, test_data, lambda_user=10.0, lambda_item=10.0):


    global_mean, user_biases, item_biases = compute_all_biases(train_data, lambda_user=lambda_user, lambda_item=lambda_item)

  
    train_with_biases = train_data.join(user_biases, on="user_id", how="left") \
                                .join(item_biases, on="anime_id", how="left")
    
    test_with_biases = test_data.join(user_biases, on="user_id", how="left") \
                              .join(item_biases, on="anime_id", how="left")
                              
    # Fill missing biases in test set with 0 (maybe change idk if correct and really nesecary)
    train_with_biases = train_with_biases.fillna(0, subset=["user_bias", "item_bias"])
    test_with_biases = test_with_biases.fillna(0, subset=["user_bias", "item_bias"])

    # Score - mu - bx - bi (score - gobal_bias - user_bias - item_bias
    train_with_biases = train_with_biases.withColumn(
        "centered_score", 
        F.col("score") - global_mean - F.col("user_bias") - F.col("item_bias")
    )

    # Train ALS 
    als_bias = ALS(
        userCol="user_id",
        itemCol="anime_id",
        ratingCol="centered_score",   
        coldStartStrategy="drop"
    )

    # check the best parameters for the model
    param_grid = ParamGridBuilder() \
        .addGrid(als_bias.rank, [100]) \
        .addGrid(als_bias.regParam, [0.4, 0.8]) \
        .build()
    
    bias_evaluator = RegressionEvaluator(
        metricName="rmse",
        labelCol="centered_score",   
        predictionCol="prediction"
    )

    crossval = CrossValidator(
        estimator=als_bias,
        estimatorParamMaps=param_grid,
        evaluator=bias_evaluator, 
        numFolds=3
    )


    cv_model_biased = crossval.fit(train_with_biases)
    best_biased_model = cv_model_biased.bestModel # best model found

    # Make predictions on the test data
    predictions_with_bias = best_biased_model.transform(test_with_biases)

    # Reconstruct the real prediction (Slide Math: Prediction + mu + bx + bi)
    final_predictions = predictions_with_bias.withColumn(
        "final_prediction", 
        F.col("prediction") + global_mean + F.col("user_bias") + F.col("item_bias")
    )

    # Evaluate the rmse in function of the score
    final_evaluator = RegressionEvaluator(
        metricName="rmse",
        labelCol="score",               
        predictionCol="final_prediction" 
    )
    
    final_rmse = final_evaluator.evaluate(final_predictions)
    print(f"RMSE (With all bias + hyper para): {final_rmse}")

    best_rank = best_biased_model.rank
    best_reg_param = best_biased_model._java_obj.parent().getRegParam()

    print(f"Best Rank chosen: {best_rank}")
    print(f"Best RegParam chosen: {best_reg_param}")
    print(f"Lambda User: {lambda_user} | Lambda Item: {lambda_item}")

    return best_biased_model, final_predictions , final_rmse

In [11]:
# data with profiles with 3 review or less removed
# cleaned_data = clean_sparse_data(final_data, min_reviews=3)


# train_data, test_data = cleaned_data.randomSplit([0.8, 0.2], seed=42)

# 3. Train the model
# best_model = train_and_evaluate_als(train_data, test_data)

In [13]:
def Final_train_model(min_user_reviews=3, max_user_reviews=float('inf'), min_anime_reviews=3, max_anime_reviews=float('inf'), lambda_user=10.0, lambda_item=10.0):
    final_data = clean_sparse_data(
        start_data, 
        min_user_reviews=min_user_reviews, 
        max_user_reviews=max_user_reviews,
        min_anime_reviews=min_anime_reviews, 
        max_anime_reviews=max_anime_reviews )
    
    train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)
    best_model, final_predictions, final_rmse  = train_and_evaluate_als(train_data, test_data, lambda_user=lambda_user, lambda_item=lambda_item)
    return best_model

    

In [14]:
# model = Final_train_model(min_user_reviews=0, min_anime_reviews=0)

In [15]:
#only used for the plots

def plot_train_model(min_user_reviews=3, max_user_reviews=float('inf'), min_anime_reviews=3, max_anime_reviews=float('inf'), lambda_user=10.0, lambda_item=10.0):
    final_data = clean_sparse_data(
        start_data, 
        min_user_reviews=min_user_reviews, 
        max_user_reviews=max_user_reviews,
        min_anime_reviews=min_anime_reviews, 
        max_anime_reviews=max_anime_reviews )
    
    train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)
    best_model, final_predictions , final_rmse  = train_and_evaluate_als(train_data, test_data, lambda_user=lambda_user, lambda_item=lambda_item) 
    return best_model,  final_predictions, final_rmse

In [ ]:
best_model.write().overwtite().save('/home/jovyan/work/models/user-item')